In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import mlflow

from data.scripts.shared.text_metrics import compute_author_metrics
import nltk

from scipy import stats

from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:2000"
MLFLOW_EXPERIMENT_NAME = "ruslit_metrics_analysis"
MLFLOW_RUN_NAME = "first run"

MLFLOW_PREPROCESS_RUN_ID = "406f39f4f7844e10b58b1b508d6f2bc7"
MLFLOW_DVC_HASH = "13661539f0bb85d192838c07b228b598.dir"
MLFLOW_GIT_HASH = "861ccef2e23e419887a68c0020becc254d0c6e57"

DF_PATH_NAME = "/Users/pgdev/PycharmProjects/rubert_author_attribution/data/data/balanced_lit.csv"

METRICS_LIST = [
    'avg_sent_len_words',
    'avg_word_len_chars',

    'mtld',

    'avg_tree_depth',
    'passive_ratio',
    'func_word_ratio',
    'avg_clauses_per_sent',
    'noun_verb_ratio',

    'compression_ratio',

    'flesch_reading_ease',
    'gunning_fog',
]
RANDOM_STATE = 67

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
ru_stopwords = set(nltk.corpus.stopwords.words("russian"))

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
#if mlflow.active_run() is None:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.start_run(run_name=MLFLOW_RUN_NAME)
mlflow.log_param("dataset_path", DF_PATH_NAME)

mlflow.set_tag("upstream.preprocess_run_id", MLFLOW_PREPROCESS_RUN_ID)
mlflow.set_tag("dataset.dvc_md5", MLFLOW_DVC_HASH)
mlflow.set_tag("dataset.git_commit", MLFLOW_GIT_HASH)

In [ ]:
df = pd.read_csv(DF_PATH_NAME, delimiter=',')
df.columns=["author","text", 'source_id']
df.head()

In [ ]:
category_group = df.groupby(['author'])
grouped = df['author'].value_counts()

grouped

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.barplot(grouped, alpha=0.8, ax=ax)
ax.set_ylabel('Number of Occurrences', fontsize=12)
ax.set_xlabel('Author Name', fontsize=12)
ax.tick_params(axis='x', rotation=90)
fig.tight_layout()

In [ ]:
mlflow.log_figure(fig, "plots/author_chunk_count.png")

In [ ]:
from joblib import Parallel, delayed

author_metrics = Parallel(n_jobs=-1)(
    delayed(compute_author_metrics)(group, author)
    for author, group in df.groupby("author")
)

author_df = pd.DataFrame(author_metrics).set_index("author")

In [ ]:
author_df.head()

In [ ]:
author_df.to_csv("lit_author_metrics.csv")

In [ ]:
dataset = mlflow.data.from_pandas(author_df, name="metrics_df")

mlflow.log_input(dataset, context="metrics_df")

In [ ]:
mlflow.log_artifact("lit_author_metrics.csv", artifact_path="data")

In [ ]:
_, p_normal_s = stats.shapiro(author_df['avg_tree_depth_mean'].dropna())
if p_normal_s > 0.05:
    print(f"по метрике {'avg_tree_depth_mean'} данные распределены нормально ({p_normal_s})")
else:
    print(f"по метрике {'avg_tree_depth_mean'} данные НЕ распределены нормально ({p_normal_s})")

In [ ]:
for metric in METRICS_LIST:
    col_mean = f'{metric}_mean'
    col_median = f'{metric}_median'

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.violinplot(data=author_df, y=col_mean, ax=axes[0])
    axes[0].set_title(col_mean)
    sns.violinplot(data=author_df, y=col_median, ax=axes[1])
    axes[1].set_title(col_median)

    fig.suptitle(f'Распределение {metric} по авторам', fontsize=14)
    fig.tight_layout()

    display(fig)
    mlflow.log_figure(fig, f"plots/{metric}_violin.png")
    plt.close(fig)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def create_author_similarity_matrix(df, ngram_range):
    """
    Создать матрицу сходства между авторами на основе их n-грамм
    """
    # Создаем корпус текстов для каждого автора
    author_texts = []
    authors = df['author'].unique()

    for author in authors:
        author_text = ' '.join(df[df['author'] == author]['text'].tolist())
        author_texts.append(author_text)

    # Векторизуем n-граммы
    vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=500)
    ngram_matrix = vectorizer.fit_transform(author_texts)

    # Вычисляем косинусное сходство
    similarity_matrix = cosine_similarity(ngram_matrix)

    return similarity_matrix, authors

sample_df = df.groupby('author').sample(n=250, random_state=RANDOM_STATE)
similarity_matrix, authors = create_author_similarity_matrix(sample_df, (3,5))

fig, ax = plt.subplots(figsize=(75, 20))
sns.heatmap(similarity_matrix,
            xticklabels=authors,
            yticklabels=authors,
            annot=True,
            vmin=0, vmax=1)
ax.set_title('Сходство авторов на основе n-грамм')

In [ ]:
mlflow.log_figure(fig, f"plots/sim_matrix_3_5_ngrams.png")

In [ ]:
def get_top_ngrams_by_author(df, ngram_range, top_k):
    author_ngrams = {}

    for author in df['author'].unique():
        author_texts = ' '.join(df[df['author'] == author]['text'].tolist())

        vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words=None)
        ngram_matrix = vectorizer.fit_transform([author_texts])

        ngram_counts = ngram_matrix.toarray().flatten()
        ngram_features = vectorizer.get_feature_names_out()

        top_indices = ngram_counts.argsort()[-top_k:][::-1]

        author_ngrams[author] = {
            'ngrams': [ngram_features[i] for i in top_indices],
            'counts': [ngram_counts[i] for i in top_indices]
        }

    return author_ngrams

sample_df = df.groupby('author').sample(n=290, random_state=RANDOM_STATE)
ngrams_dict = get_top_ngrams_by_author(sample_df, ngram_range=(3,5), top_k=10)

In [ ]:
mlflow.log_dict(ngrams_dict, "data/ngrams_3_5_dict.json")

In [ ]:
def plot_top_ngrams(ngrams_dict, author):
    data = ngrams_dict[author]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(data['ngrams'], data['counts'])
    ax.set_xlabel('Частота')
    ax.set_title(f'Топ n-грамм для автора {author}')
    ax.invert_yaxis()
    fig.tight_layout()

    display(fig)
    mlflow.log_figure(fig, f"plots/3_5_top_ngrams_{author}.png")
    plt.close(fig)

for author in ngrams_dict.keys():
    plot_top_ngrams(ngrams_dict, author)

In [ ]:
def find_unique_ngrams(df, ngram_range=(3,5)):

    authors = df['author'].unique()
    all_ngrams = {}

    for author in authors:
        author_text = ' '.join(df[df['author'] == author]['text'].tolist())
        vectorizer = CountVectorizer(ngram_range=ngram_range)
        ngram_matrix = vectorizer.fit_transform([author_text])
        ngram_counts = ngram_matrix.toarray().flatten()
        ngram_features = vectorizer.get_feature_names_out()

        all_ngrams[author] = set(ngram_features[ngram_counts > 0])

    unique_ngrams = {}
    for author in authors:
        other_authors_ngrams = set().union(*[all_ngrams[a] for a in authors if a != author])
        unique_ngrams[author] = all_ngrams[author] - other_authors_ngrams
        print(f"Автор {author}: {len(unique_ngrams[author])} уникальных n-грамм")
        print(f"Примеры: {list(unique_ngrams[author])[:10]}\n")

    return unique_ngrams

find_unique_ngrams(sample_df)

In [ ]:
def is_data_normal(df, metric_name: str, a: float = 0.05) -> bool:
    _, p_normal_s = stats.shapiro(df[metric_name].dropna())
    if p_normal_s > 0.05:
        print(f"Данные по метрике {metric_name} нормально распределены ({p_normal_s})")
        return True
    else:
        print(f"Данные по метрике {metric_name} НЕ нормально распределены ({p_normal_s})")
        return False

In [ ]:
normal_dist_dict = {}
for metric in METRICS_LIST:
    flag1 = is_data_normal(author_df, metric + '_mean')
    flag2 = is_data_normal(author_df, metric + '_median')
    normal_dist_dict[metric + '_mean'] = flag1
    normal_dist_dict[metric + '_median'] = flag2
    print()
mlflow.log_dict(normal_dist_dict, "data/normal_dist_dict.json")

In [ ]:
mlflow.end_run()